In [23]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LeakyReLU, BatchNormalization, Dropout
import numpy as np

In [ ]:
#Creamos solamente el generador
generador = Sequential()
    
#El tamaño del vector aleatorio (semilla del generador) indica la complejidad de los 
#datos generados:
# A mayor tamaño: más complejos son los datos sintetizados, y más difícil el entrenamiento
# A menor tamaño: más simple son los datos sintetizados y más fácil de entrenar.
tam_semilla_aleatoria = 30 #para generar matrículas es suficiente

# Capa densa inicial
generador.add(Dense(64, input_dim=tam_semilla_aleatoria))
generador.add(LeakyReLU(alpha=0.2))

# Capa oculta
generador.add(Dense(128))
generador.add(LeakyReLU(alpha=0.2))
    
# Capa oculta
generador.add(Dense(64))
generador.add(LeakyReLU(alpha=0.2))
generador.add(BatchNormalization(momentum=0.1))
    
# Capa de salida con 7 números
generador.add(Dense(7, activation=None)) #función de activación identidad

generador.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_4 (Dense)             (None, 64)                1984      
                                                                 
 leaky_re_lu_3 (LeakyReLU)   (None, 64)                0         
                                                                 
 dense_5 (Dense)             (None, 128)               8320      
                                                                 
 leaky_re_lu_4 (LeakyReLU)   (None, 128)               0         
                                                                 
 dense_6 (Dense)             (None, 64)                8256      
                                                                 
 leaky_re_lu_5 (LeakyReLU)   (None, 64)                0         
                                                                 
 dense_7 (Dense)             (None, 7)                

In [25]:
#Vamos a crear un ruido aleatorio y probar el generador
ruido_aleatorio = np.random.uniform( size=(1,tam_semilla_aleatoria))
print('ruido aleatorio', ruido_aleatorio)

muestra = generador.predict(ruido_aleatorio)
print('muestra generada:', muestra)
#Lo ideal sería ver la matrícula no con números, si no en formato 9999XXX

ruido aleatorio [[0.96024188 0.8744061  0.82867271 0.79752984 0.62000653 0.93151276
  0.57874295 0.36102212 0.07969408 0.50816752 0.15048242 0.7542947
  0.77298328 0.41872671 0.91819257 0.38256365 0.02845724 0.72800455
  0.43482066 0.9037418  0.94667876 0.02043104 0.09666614 0.60224551
  0.90810108 0.2138059  0.26494436 0.0392283  0.86431388 0.1376715 ]]
1/1 [==============================] - 0s 45ms/step
muestra generada: [[-0.12656407  0.00571473  0.0216636  -0.5248613  -0.0855223   0.2311938
  -0.7453749 ]]


Vamos a implementar un mecanismo para poder ver las matrículas con el formato 9999XXX

In [26]:
import string
#letras es un diccionario de la forma: 0:A, 1:B, ...
letras = {indice: letra for indice, letra in enumerate(string.ascii_uppercase)
              if letra not in ['A', 'E', 'I', 'O', 'U', 'Q']}
letras

{1: 'B',
 2: 'C',
 3: 'D',
 5: 'F',
 6: 'G',
 7: 'H',
 9: 'J',
 10: 'K',
 11: 'L',
 12: 'M',
 13: 'N',
 15: 'P',
 17: 'R',
 18: 'S',
 19: 'T',
 21: 'V',
 22: 'W',
 23: 'X',
 24: 'Y',
 25: 'Z'}

In [27]:
#Esta función recibe una serie de arrays con la forma de las propiedades de un dataset:
# [ datos1,
#   datos2,
#   ....   ]
# donde datos9 es un array de 6 números, que tendremos que transformas en el formato de
# una matrícula. Por ejemplo:
# [ 1 2 3 18 12 4 2]
#   123-LDB
#el 18 no es un número que aparezca en una matrícula, por eso se pone un guión (-) 

def formato_matricula(X, letras):
    matriculas = np.array([])
    for muestra in X:
        matricula = ''
        #números
        for i in muestra[0:4]:
            if 0<=i and i <=9:
                matricula += str(round(i))
            else:
                matricula += '-'
        #letras
        for i in muestra[4:]:
            i = round(i)

            if i in letras:
                matricula += letras[i]
            else:
                matricula += '-'

        matriculas = np.append(matriculas, [matricula])

    return matriculas

In [28]:
muestra = np.array ( [ [-1.2, 0.3, 4.5, 6.7, 4.01, 12.18, 3.9] ])
matricula = formato_matricula(muestra, letras)
print(matricula)

['-047-M-']


In [48]:
# Generamos una muestra aleatoria
ruido_aleatorio = np.random.uniform(-1, 1, (1, tam_semilla_aleatoria))
print("Ruido aleatorio:", ruido_aleatorio)

matricula = generador.predict(ruido_aleatorio)
print(matricula)
print("Matricula:", formato_matricula(matricula, letras))


Ruido aleatorio: [[-0.94670137 -0.59081418 -0.72395242 -0.15737423  0.22958295 -0.03362992
  -0.29916949 -0.12753193 -0.96366989  0.57292668  0.56426051  0.06308439
   0.58008619 -0.68477838 -0.46977047  0.88746885  0.42359879 -0.52674326
  -0.14910026  0.27867694  0.65910187 -0.89076835  0.22008396  0.8310602
  -0.9204202   0.68789553 -0.43789761  0.40387536  0.33693913  0.62535332]]
1/1 [==============================] - 0s 11ms/step
[[-0.3649909  -0.02522675  0.09005287  0.09539154  0.0805347  -0.2231325
   0.16116199]]
Matricula: ['--00---']
